# 02 — Check Orthogonality & Select Stimuli

This notebook:
1. Verifies the correlation between load and abstractness scores (target: |r| < 0.3)
2. Selects a balanced stimulus set by sampling from 3×3 tertile grid
3. Computes VIF to check for multicollinearity
4. Visualizes the selected stimulus space

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import pearsonr
from sklearn.linear_model import LinearRegression

sns.set_style('whitegrid')
sns.set_context('notebook')
%matplotlib inline

## Load Scores

In [ ]:
df = pd.read_csv('data/nsd_image_scores.csv')
print(f'Total images: {len(df)}')
print(f'Columns: {list(df.columns)}')

# Filter to images with both scores
both_valid = df['load_z'].notna() & df['abstractness_z'].notna()
df_valid = df[both_valid].copy()
print(f'Images with both scores: {len(df_valid)}')

## Corpus-Level Orthogonality Check

In [ ]:
r, p = pearsonr(df_valid['load_z'], df_valid['abstractness_z'])
print(f'Pearson r(load_z, abstractness_z) = {r:.4f}')
print(f'p-value = {p:.2e}')
print(f'Passes |r| < 0.3 check: {abs(r) < 0.3}')

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Joint scatter
axes[0].scatter(df_valid['load_z'], df_valid['abstractness_z'],
                alpha=0.03, s=2, color='teal')
axes[0].set_xlabel('Load (z-scored entropy)')
axes[0].set_ylabel('Abstractness (z-scored)')
axes[0].set_title(f'Full corpus: r = {r:.3f}')
axes[0].axhline(0, color='gray', ls='--', lw=0.5)
axes[0].axvline(0, color='gray', ls='--', lw=0.5)

# Hexbin for density
hb = axes[1].hexbin(df_valid['load_z'], df_valid['abstractness_z'],
                     gridsize=50, cmap='viridis', mincnt=1)
axes[1].set_xlabel('Load (z)')
axes[1].set_ylabel('Abstractness (z)')
axes[1].set_title('Density (hexbin)')
plt.colorbar(hb, ax=axes[1], label='Count')

# Correlation matrix of all numeric regressors
reg_cols = [c for c in ['load_z', 'abstractness_z', 'mean_luminance_z', 'rms_contrast_z']
            if c in df_valid.columns]
corr_mat = df_valid[reg_cols].corr()
sns.heatmap(corr_mat, annot=True, fmt='.3f', cmap='RdBu_r',
            center=0, vmin=-1, vmax=1, ax=axes[2])
axes[2].set_title('Regressor Correlations')

plt.tight_layout()
plt.show()

## Stimulus Selection

Divide each axis into tertiles and sample ~30 images per cell (9 cells → ~270 images).

In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), 'scripts'))
from select_stimuli import assign_tertile, check_orthogonality, compute_vif

N_PER_CELL = 30
SEED = 42
MAX_COLLINEARITY = 0.3

df_valid['load_tertile'] = assign_tertile(df_valid['load_z'])
df_valid['abstract_tertile'] = assign_tertile(df_valid['abstractness_z'])

print('Available images per cell:')
cell_counts = df_valid.groupby(['load_tertile', 'abstract_tertile']).size()
print(cell_counts.unstack(fill_value=0))

In [ ]:
rng = np.random.default_rng(SEED)
best_selection = None
best_r = 1.0

for attempt in range(50):
    selected_parts = []
    for (lt, at), group in df_valid.groupby(['load_tertile', 'abstract_tertile']):
        n_sample = min(N_PER_CELL, len(group))
        sampled = group.sample(n=n_sample, random_state=rng.integers(1e9))
        selected_parts.append(sampled)

    df_sel = pd.concat(selected_parts, ignore_index=True)
    r_sel, passes = check_orthogonality(
        df_sel['load_z'].values, df_sel['abstractness_z'].values, MAX_COLLINEARITY
    )

    if abs(r_sel) < abs(best_r):
        best_r = r_sel
        best_selection = df_sel

    if passes:
        print(f'PASSED on attempt {attempt+1}: r = {r_sel:.4f}')
        break
else:
    print(f'Best achieved: r = {best_r:.4f}')

df_selected = best_selection
print(f'\nSelected {len(df_selected)} images')

In [ ]:
# VIF check
X = df_selected[['load_z', 'abstractness_z']].values
vifs = compute_vif(X)
print(f'VIF(load_z) = {vifs[0]:.3f}')
print(f'VIF(abstractness_z) = {vifs[1]:.3f}')
print('VIF < 5 indicates acceptable collinearity')

## Visualize Selected Set

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Selected images in score space
colors = {'low': 'C0', 'mid': 'C1', 'high': 'C2'}
for lt in ['low', 'mid', 'high']:
    for at in ['low', 'mid', 'high']:
        mask = (df_selected['load_tertile'] == lt) & (df_selected['abstract_tertile'] == at)
        sub = df_selected[mask]
        axes[0].scatter(sub['load_z'], sub['abstractness_z'],
                        alpha=0.6, s=20, label=f'L={lt},A={at}')

axes[0].set_xlabel('Load (z)')
axes[0].set_ylabel('Abstractness (z)')
r_final = pearsonr(df_selected['load_z'], df_selected['abstractness_z'])[0]
axes[0].set_title(f'Selected Stimuli (n={len(df_selected)}, r={r_final:.3f})')

# Cell counts heatmap
ct = df_selected.groupby(['load_tertile', 'abstract_tertile']).size().unstack(fill_value=0)
# Reorder
for lbl in ['low', 'mid', 'high']:
    if lbl not in ct.index:
        ct.loc[lbl] = 0
    if lbl not in ct.columns:
        ct[lbl] = 0
ct = ct.loc[['low', 'mid', 'high'], ['low', 'mid', 'high']]
sns.heatmap(ct, annot=True, fmt='d', cmap='YlGnBu', ax=axes[1])
axes[1].set_xlabel('Abstract Tertile')
axes[1].set_ylabel('Load Tertile')
axes[1].set_title('Images per Cell')

# Marginal distributions
axes[2].hist(df_selected['load_z'], bins=30, alpha=0.6, label='Load (z)', color='steelblue')
axes[2].hist(df_selected['abstractness_z'], bins=30, alpha=0.6, label='Abstractness (z)', color='mediumpurple')
axes[2].legend()
axes[2].set_xlabel('Z-score')
axes[2].set_title('Marginal Distributions')

plt.tight_layout()
plt.show()

## Save Selected Stimulus Set

In [ ]:
output_path = 'data/selected_image_ids.csv'
df_selected.to_csv(output_path, index=False)
print(f'Saved {len(df_selected)} selected images to {output_path}')